# Sync Video Led Vec with TTL events

## Read Video vecs

In [19]:
VID_FPS = 30
MAX_HOUR_IN_SECONDS = 60*60*4
TDT = True

In [20]:
import numpy as np

spikeglx_folder = '//path/Pixel1/1_auditory_neuropixels_BarakH/20250212_C16_T1_NP2_-10dB_g0'
video_vecs = np.load(spikeglx_folder + "/video_loc.npz")

Plot first frame

In [21]:
import matplotlib.pyplot as plt
import tdt

# import matplotlib as mpl
# mpl.use('TkAgg')

if TDT:
    data = tdt.read_block(spikeglx_folder + '/BARAK-250212-120932')
    led_time_vec = data.epocs.Cam1.onset
    loc_time_vec = data.epocs.Cam1.onset
    
else:
    vid_sr = VID_FPS
    led_time_vec = np.linspace(0, len(video_vecs['led_info']), len(video_vecs['led_info'])) / vid_sr
    loc_time_vec = np.linspace(0, len(video_vecs['location_com']), len(video_vecs['location_com'])) / vid_sr


plt.figure(figsize=(10,10))
ax1 = plt.subplot2grid((1, 1), (0, 0), rowspan=1)
ax1.imshow(video_vecs['first_frame'])
plt.figure(figsize=(10,10))
ax2 = plt.subplot2grid((2, 1), (0, 0), rowspan=1)
ax2.plot(led_time_vec, video_vecs['led_info'][:len(led_time_vec),:])
ax2.axis(xmin=0,xmax=2*60)
ax3 = plt.subplot2grid((2, 1), (1, 0), rowspan=1)
ax3.plot(loc_time_vec, video_vecs['location_com'][:len(loc_time_vec),:])
ax3.axis(xmin=0,xmax=5*60)

plt.show(block=False)



## Read TTL vec

In [22]:
from Global_functions import read_file_by_time_steps
from pathlib import Path
import numpy as np
import re
import os

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                print(f"File found: {file_path}")
    return matching_files

tstep = 60
t_vec = np.arange(0, MAX_HOUR_IN_SECONDS, tstep)

binFullPath_NI = Path(find_files(spikeglx_folder, r'.*\.nidq\.bin$')[0])
TTL_NI, _ = read_file_by_time_steps(binFullPath_NI, t_vec, tstep, [4])
TTL_NI = np.concatenate(TTL_NI, axis=1)
tNI = np.linspace(t_vec[0], t_vec[-1] + tstep, TTL_NI.shape[1])
TTL_vec = (TTL_NI[0,:]-min(TTL_NI[0,:]))/(max(TTL_NI[0,:])-min(TTL_NI[0,:]))
led_vec = video_vecs['led_info'][:,0]
led_vec = (led_vec-min(led_vec))/(max(led_vec)-min(led_vec))

## Sync Led with TTL

In [23]:
led_vec[led_vec<0.4] = 0
TTL_vec_filt = TTL_vec.copy()
TTL_vec_filt[TTL_vec<0.5] = 0
# led_shifted = led_vec.copy()
# t_pad = 2 * vid_sr
# led_shifted = np.pad(led_shifted, (t_pad,0), mode='constant', constant_values=0)
len_vec = VID_FPS * 50

led_flipped = led_vec[0:len_vec]
led_flipped = led_flipped[::-1]
t_vec = led_time_vec[0:len_vec]
t_conv = t_vec-t_vec[len_vec//2]

TTL_NI_led = np.interp(t_vec, tNI[tNI<=max(t_vec)], TTL_vec_filt[tNI<=max(t_vec)])

conv_res = np.convolve(TTL_NI_led, led_flipped, 'same')
print(f'max {max(conv_res)} ind {np.argmax(conv_res)} dt {t_conv[np.argmax(conv_res)]} sec')
t_vec_fixed = t_vec + t_conv[np.argmax(conv_res)]

plt.figure(figsize=(10,10))
plt.subplot(211)
plt.plot(t_conv, conv_res, label='Conv')
plt.subplot(212)
plt.plot(t_vec, led_vec[0:len_vec], label='OG_led')
plt.plot(tNI[tNI<=max(t_vec)], TTL_vec_filt[tNI<=max(t_vec)], label='TTL')
plt.plot(t_vec_fixed, led_vec[0:len_vec] + 1, label='fixed_led')
plt.legend()
# ax2.plot(tNI, TTL_vec, label='TTL')
# # ax2.axis(xmin=0,xmax=2*60)
# # ax2 = plt.subplot2grid((2, 1), (1, 0), rowspan=1)
# # ax2.plot(led_time_vec, led_vec, label='Led')
# # ax2.plot(led_time_vec, led_vec_plot, label='Led shifted')
# ax2.axis(xmin=0,xmax=5*60)
# ax2.legend(loc='upper right')
plt.show(block=False)

## Save Video data after Sync

In [24]:
save_path = spikeglx_folder + "/video_loc_sync.npz"
np.savez(save_path, led_info=video_vecs['led_info'], location_com=video_vecs['location_com'], first_frame=video_vecs['first_frame'], t_vec=led_time_vec + t_conv[np.argmax(conv_res)])